# JACS 2021 LEDC Reaction Network — Walkthrough

This notebook reproduces, at a teaching level, the pipeline of
Xie *et al.*, *J. Am. Chem. Soc.* **2021**, *143*, 13245
(DOI [10.1021/jacs.1c05807](https://doi.org/10.1021/jacs.1c05807)).

We follow the paper's four stages:

1. **Species generation** — fragmentation + recombination from seed molecules
   (Sec. 2.1).
2. **Reaction enumeration** — all reactions with chemical distance $CD\le 5$
   (Sec. 2.2). Here we use $CD\le 3$ for runtime.
3. **Free-energy assignment** — DFT in the paper, mock values here.
4. **Pathfinding** — softplus cost graph + Dijkstra/Yen K-shortest paths
   (Sec. 2.3) to the target **LEDC**.

Activate the env first:
```bash
conda activate jacs2021
jupyter lab
```

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from src.molecule import MoleculeGraph, dedupe
from src.fragrec  import n_step_fragment, recombine_pool
from src.network  import build_species_pool, enumerate_reactions, chemical_distance
from src.pathfind import softplus_cost, build_pathfinding_graph, k_shortest_paths
from src.thermo   import MU_E_LI_METAL, reaction_dG
from data.seed_species import SEEDS, make_LEDC
from src.run_demo import mock_free_energy

## 1. Seed molecules

The electrolyte starting pool: ethylene carbonate (EC), water (trace), and
Li⁺. The target is LEDC.

In [2]:
seeds = [SEEDS['EC'], SEEDS['H2O'], SEEDS['Li+']]
target = make_LEDC()
for m in seeds + [target]:
    print(f'{m.name:6s}  formula={m.formula}  charge={m.charge:+d}  '
          f'atoms={m.graph.number_of_nodes()}  bonds={m.graph.number_of_edges()}')

EC      formula=C3H4O3  charge=+0  atoms=10  bonds=10
H2O     formula=H2O  charge=+0  atoms=3  bonds=2
Li+     formula=Li  charge=+1  atoms=1  bonds=0
LEDC    formula=C4H4Li2O6  charge=+0  atoms=16  bonds=15


## 2. Fragmentation + recombination

`build_species_pool` does $n$-step single-bond fragmentation, then pairwise
recombination (1-bond formation between distinct fragments), filtering
endergonic recombinants via the BonDNet stand-in (`bde_model.py`).

In [3]:
raw_pool = build_species_pool(seeds, n_frag_steps=1, keep_endergonic_recomb=False)
print(f'raw pool size: {len(raw_pool)}')

# Trim like run_demo does so the rest of the notebook is fast.
MAX_HEAVY, MAX_POOL = 12, 80
pool = [m for m in raw_pool
        if sum(1 for _, d in m.graph.nodes(data=True) if d['element'] != 'H') <= MAX_HEAVY]
if not any(target.is_isomorphic(m) for m in pool):
    pool.append(target)
pool = dedupe(pool)[:MAX_POOL]
if not any(target.is_isomorphic(m) for m in pool):
    pool.append(target)
    pool = dedupe(pool)
print(f'trimmed pool size: {len(pool)}')

# Show a sample
for m in pool[:10]:
    print(' ', m.formula, m.name or '')

raw pool size: 2


AttributeError: 'list' object has no attribute 'graph'

## 3. Free energies

In the real paper these come from ωB97X-V/def2-TZVPPD/SMD single points on
ωB97X-D3/def2-SVPD geometries. Here we use a transparent mock model that
rewards C–O / O–H / Li–O bonds and is reproducible.

In [ ]:
G = {m.canonical_hash(): mock_free_energy(m) for m in pool}
values = np.array(list(G.values()))
plt.hist(values, bins=30, color='steelblue', edgecolor='k', alpha=0.8)
plt.xlabel('mock $G$ (eV)'); plt.ylabel('count'); plt.title('Free-energy distribution');

## 4. Reaction enumeration with $CD\le 3$

We enumerate concerted reactions with up to two reactants and two products;
the chemical-distance lower bound (edge-multiset symmetric difference) is the
main pruning device, mirroring the paper's CD constraint.

In [ ]:
reactions = enumerate_reactions(pool, G, max_bond_changes=3,
                                max_reactants=2, max_products=2,
                                mu_e=MU_E_LI_METAL)
print(f'{len(reactions)} reactions, μ_e = {MU_E_LI_METAL:+.2f} eV (Li/Li⁺)')

dGs = np.array([r.dG for r in reactions])
plt.hist(dGs, bins=60, color='salmon', edgecolor='k', alpha=0.8)
plt.axvline(0, color='k', lw=0.5)
plt.xlabel(r'$\Delta G$ (eV)'); plt.ylabel('count'); plt.title('Reaction $\Delta G$ spectrum');

## 5. Pathfinding to LEDC

Reactions become directed edges with weight $w = \mathrm{softplus}(\Delta G)$;
a virtual SOURCE node connects to the starting pool. Yen's algorithm yields
the K shortest paths.

In [ ]:
starting = {m.canonical_hash() for m in seeds}
pf = build_pathfinding_graph(reactions, starting_pool=starting, scale=0.5)
print(f'path graph: {pf.number_of_nodes()} nodes, {pf.number_of_edges()} edges')

paths = k_shortest_paths(pf, target.canonical_hash(), k=5)
name = {m.canonical_hash(): (m.name or m.formula) for m in pool}
for i, p in enumerate(paths, 1):
    print(f'\n--- Path {i} ---')
    print(p.pretty(name))

## 6. What is *not* reproduced

* DFT free energies (replace `mock_free_energy` with QChem outputs).
* Real BonDNet (replace `bde_model.predict_recomb_dG`).
* The full $CD\le 5$ network with ~9 M reactions (raise `max_bond_changes`
  and `MAX_POOL_SIZE`, expect orders-of-magnitude longer runtimes).
* MILP atom mapping for *exact* CD; we use an edge-multiset lower bound.
* Solvation + electrochemical corrections beyond the constant $\mu_e$ shift.